# Director comparisons from selected strain-coupling simulations

Default: no static exports, no automatic animation generation, director-only panels at higher glyph density. Each GIF compares 1-3 selected simulations in one row on X+u at scale 1, with a shared S scale and spatial bounds fixed over the full saved history. Directors come from Q; no extra rotation is introduced. Changing glyph density does not refine the numerical mesh.

The configured manifest contains verified F IDs 27-32. Change `DIRECTOR_COMPARISONS` and the manifest for other F/AC groups; optional `DIRECTOR_CASE_CHECKS` protect the example parameter tuples. `SELECTION_MODE="original"` retains A-H loading. `INCLUDE_MECHANICAL_FIELDS=True` restores the original three-field movies; `GENERATE_STATIC_SNAPSHOTS=True` enables static figures. Existing outputs are never overwritten.

Low-S warning percentages integrate the region below the threshold in the saved P1 S interpolant over reference triangles. Glyph suppression remains conservatively based on low-S bins. Mechanical flags describe the reconstructed P1 displacement, not a new verification of the original P2 solution. Large strain flags are local screening results; inverted previews are nonphysical geometries. All saved timestamps are used without state interpolation; unavailable partner frames are explicitly blank.


In [21]:
from pathlib import Path
from datetime import datetime, timezone
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from matplotlib.collections import LineCollection, PolyCollection
from matplotlib.colors import Normalize, LogNorm
import matplotlib.patheffects as pe

ROOT = Path.cwd()  # Set explicitly if Jupyter starts outside the repository.
# Choose a preset, then restart the kernel and Run All.
VISUALIZATION_PRESET = "local_directors"  # local_directors, AC, F, F_activity, eps030_contractile, eps030_positive, eps030_zero, original, saved_manifest
MANIFEST_ROOT = ROOT / "outputs/strain_coupling/selections"
# For saved_manifest, choose any existing selection file; each selected run gets its own movie.
SAVED_MANIFEST = MANIFEST_ROOT / "refined_F_tend5p0/selection_20261103T141426_182668Z.json"
SELECTION_MODE = "custom"
SELECTION_REPORT = None
SELECTION_MANIFEST = MANIFEST_ROOT / "refined_AC_tend5p0/selection_20261105T120205_788848Z.json"
CANDIDATE_CSV = None  # None finds the latest verified_candidates.csv, not the sweep itself.
SELECTED_LABELS = list("ABCDEFGH")
REQUESTED_TIMES = [0.0, 2.5, 5.0]
INCLUDE_MAX_RESPONSE_TIME = True
COMPARISON_TIME = 2.5
GEOMETRY = "deformed"  # Alternative: "reference"
ALLOW_INVERTED_PREVIEW = True  # Explicitly labelled nonphysical mathematical previews.
DEFORM_SCALE = 1.0      # Never automatically rescaled; non-unit values are labelled.
GENERATE_STATIC_SNAPSHOTS = False
INCLUDE_MECHANICAL_FIELDS = False
GENERATE_ANIMATIONS = True
DIRECTOR_BINS_X, DIRECTOR_BINS_Y = (20, 60) if VISUALIZATION_PRESET == "local_directors" else (10, 30)
NBINS_X, NBINS_Y = DIRECTOR_BINS_X, DIRECTOR_BINS_Y
DIRECTOR_COMPARISONS = {
    "WeakActivity_eps0.025_gamma0": ['AC_036', 'AC_037', 'AC_038'],
    "WeakActivity_eps0.025_gamma1": ['AC_039', 'AC_040', 'AC_041'],
    "WeakActivity_eps0.025_gamma2": ['AC_042', 'AC_043', 'AC_044'],
    "Coupling_eps0.025_zeta0_gamma0-1-2": ['AC_000', 'AC_009', 'AC_015'],
    "Coupling_eps0.025_zeta0_gamma0.25-0.5-1.5": ['AC_003', 'AC_006', 'AC_012'],
    "Coupling_eps0.025_zeta-0.3": ['AC_004', 'AC_010', 'AC_016'],
    "Coupling_eps0.025_zeta-0.6": ['AC_008', 'AC_014', 'AC_017'],
    "Activity_eps0.05_gamma0.5": ['AC_024', 'AC_025', 'AC_026'],
    "Activity_eps0.05_gamma1": ['AC_027', 'AC_028', 'AC_029'],
    "Activity_eps0.05_gamma2": ['AC_033', 'AC_034', 'AC_035'],
    "Strain_gamma2_zeta-0.6": ['AC_019', 'AC_022', 'AC_031'],
    "strain_g2_zm06": ['AC_017', 'AC_035'],
}
# Combined-report IDs and signed controls, checked again before export.
DIRECTOR_CASE_CHECKS = {
    "AC_000": ("AC", 0, 0.025, 0, 0),
    "AC_003": ("AC", 3, 0.025, 0.25, 0),
    "AC_004": ("AC", 4, 0.025, 0.25, -0.3),
    "AC_006": ("AC", 6, 0.025, 0.5, 0),
    "AC_008": ("AC", 8, 0.025, 0.5, -0.6),
    "AC_009": ("AC", 9, 0.025, 1, 0),
    "AC_010": ("AC", 10, 0.025, 1, -0.3),
    "AC_012": ("AC", 12, 0.025, 1.5, 0),
    "AC_014": ("AC", 14, 0.025, 1.5, -0.6),
    "AC_015": ("AC", 15, 0.025, 2, 0),
    "AC_016": ("AC", 16, 0.025, 2, -0.3),
    "AC_017": ("AC", 17, 0.025, 2, -0.6),
    "AC_019": ("AC", 19, 0.05, 0, -0.3),
    "AC_022": ("AC", 22, 0.05, 0.25, -0.3),
    "AC_024": ("AC", 24, 0.05, 0.5, 0),
    "AC_025": ("AC", 25, 0.05, 0.5, -0.3),
    "AC_026": ("AC", 26, 0.05, 0.5, -0.6),
    "AC_027": ("AC", 27, 0.05, 1, 0),
    "AC_028": ("AC", 28, 0.05, 1, -0.3),
    "AC_029": ("AC", 29, 0.05, 1, -0.6),
    "AC_031": ("AC", 31, 0.05, 1.5, -0.3),
    "AC_033": ("AC", 33, 0.05, 2, 0),
    "AC_034": ("AC", 34, 0.05, 2, -0.3),
    "AC_035": ("AC", 35, 0.05, 2, -0.6),
    "AC_036": ("AC", 36, 0.025, 0, -0.05),
    "AC_037": ("AC", 37, 0.025, 0, -0.1),
    "AC_038": ("AC", 38, 0.025, 0, -0.2),
    "AC_039": ("AC", 39, 0.025, 1, -0.05),
    "AC_040": ("AC", 40, 0.025, 1, -0.1),
    "AC_041": ("AC", 41, 0.025, 1, -0.2),
    "AC_042": ("AC", 42, 0.025, 2, -0.05),
    "AC_043": ("AC", 43, 0.025, 2, -0.1),
    "AC_044": ("AC", 44, 0.025, 2, -0.2),
}
S_GLYPH_MIN = 0.1
DEFORM_STRAIN_LIMIT, DEFORM_GRAD_LIMIT = 0.1, 0.2

# Scales are selected explicitly in each plot_selected call below.
SAVE_PDFS = True
OUTPUT_BASE = ROOT / "outputs/strain_coupling/figures/sweeps/characteristic_snapshots"
# Explicitly target the reviewed export folder; never touch earlier reference figures.
EXPORT_STAMP = datetime.now(timezone.utc).strftime("selected_%Y%m%dT%H%M%S_%fZ")
OUTPUT_DIR = OUTPUT_BASE / "original" / EXPORT_STAMP
OVERWRITE_PDFS = False
COMPARISONS = {"ABC": list("ABC"), "DG": list("DG"), "EF": list("EF")}
DIAGNOSTIC_CASES = list("AEFD")
S_WARNING_TOL = 1e-6
S_EXTREME_LEVEL = 2.0  # Display annotation only, not a physical-validity criterion.

EXPECTED_CASES = {
 "A": (91,.05,0,0), "B": (81,.05,-2,-2), "C": (109,.05,2,-2),
 "D": (117,.05,4,-4), "E": (2,0,-8,-.6), "F": (189,.2,-8,0),
 "G": (187,.1,8,-4), "H": (188,.1,8,-8),
}

if VISUALIZATION_PRESET in ("F", "F_activity"):
    if VISUALIZATION_PRESET == "F":
        SELECTION_MANIFEST = MANIFEST_ROOT / "refined_F_tend5p0/selection_20261104T173522_743432Z.json"
        DIRECTOR_COMPARISONS = {
            "F_strain_comparison": ["F_012", "F_021", "F_030"],
            "F_activity_comparison": ["F_030", "F_031", "F_032"],
            "F_gamma_comparison": ["F_033", "F_030", "F_027"],
        }
    else:
        SELECTION_MANIFEST = MANIFEST_ROOT / "refined_F_tend5p0/director_groups_20261103T154323_673645Z.json"
        DIRECTOR_COMPARISONS = {
            "F_activity_gm8": ["F_030", "F_031", "F_032"],
            "F_activity_gm10": ["F_027", "F_028", "F_029"],
        }
    DIRECTOR_CASE_CHECKS = {}  # The loader verifies the manifest, report and simulation metadata.
elif VISUALIZATION_PRESET in ("eps030_contractile", "eps030_positive", "eps030_zero"):
    SELECTION_MODE = "report"
    activity = VISUALIZATION_PRESET.split("_")[1]
    tag = "ze0" if activity == "zero" else activity + "_activity"
    REPORT_STUDY_TAG = f"exploratory_eps0p30_gamm8_{tag}_tend5p0"
    REPORT_CASES = {"eps030_zeta0": (0, .30, -8, 0.)}
    if activity != "zero":
        sign, letter = (-1, "M") if activity == "contractile" else (1, "P")
        REPORT_CASES.update({f"eps030_zeta{letter}03": (1, .30, -8, sign*.3),
                             f"eps030_zeta{letter}06": (2, .30, -8, sign*.6)})
    DIRECTOR_COMPARISONS = {VISUALIZATION_PRESET: list(REPORT_CASES)}
    DIRECTOR_CASE_CHECKS = {}
elif VISUALIZATION_PRESET == "local_directors":
    SELECTION_MODE = "report"
    REPORT_STUDY_TAG = "refined_F_tend5p0"
    analysis = ROOT / "outputs/strain_coupling/analysis"
    SELECTION_REPORT = analysis / "strain_coupling_refined_F_tend5p0_20261103T120501_488511Z.csv"
    PASSIVE_REPORT = analysis / "strain_coupling_grid_tend5p0_20261026T180315_275819Z.csv"
    REPORT_CASES = {"F_021": (21, .10, -8, 0), "F_022": (22, .10, -8, -.15),
                    "F_023": (23, .10, -8, -.30), "Passive": (154, .10, 0, 0)}
    DIRECTOR_COMPARISONS = {
        f"{label}_eps0.10_gamma{gamma:g}_zeta{zeta:g}": [label]
        for label, (_, _, gamma, zeta) in REPORT_CASES.items()
    }
    DIRECTOR_CASE_CHECKS = {}  # Checked against each source report and saved metadata.
elif VISUALIZATION_PRESET == "original":
    SELECTION_MODE = "original"
elif VISUALIZATION_PRESET == "saved_manifest":
    SELECTION_MANIFEST = SAVED_MANIFEST
    DIRECTOR_COMPARISONS = {case["label"]: [case["label"]]
                            for case in json.loads(SAVED_MANIFEST.read_text(encoding="utf-8"))["cases"]}
    DIRECTOR_CASE_CHECKS = {}
elif VISUALIZATION_PRESET != "AC":
    raise ValueError(f"Unknown visualization preset: {VISUALIZATION_PRESET}")


## Verify candidates and index saved times

Directories come from the verified CSV. Metadata and parameter tuples are checked before use. H is reported separately and never sent to the field loader.


In [22]:
def resolve_folder(value):
    folder = Path(value)
    if folder.is_dir():
        return folder
    portable = str(value).replace("\\", "/")
    marker = "outputs/strain_coupling/"
    if marker not in portable:
        raise FileNotFoundError(value)
    folder = ROOT / marker / portable.split(marker, 1)[1]
    if not folder.is_dir():
        raise FileNotFoundError(folder)
    return folder

def resolve_selection_file(value):
    path = Path(value)
    if not path.is_absolute():
        path = ROOT / path
    if path.is_file():
        return path
    portable = str(value).replace("\\", "/")
    marker = "outputs/strain_coupling/"
    if marker in portable:
        path = ROOT / marker / portable.split(marker, 1)[1]
    if not path.is_file():
        raise FileNotFoundError(path)
    return path


def load_custom_selection(value):
    import hashlib, re
    if value is None:
        raise ValueError("Paste SELECTION_MANIFEST from the phase notebook's selection cell")
    path = resolve_selection_file(value)
    manifest = json.loads(path.read_text(encoding="utf-8"))
    study = manifest.get("study")
    tag = manifest.get("study_tag", "")
    if manifest.get("schema_version") != 1 or study not in ("F", "AC") or not tag.startswith(f"refined_{study}_tend"):
        raise ValueError("Unsupported or inconsistent selection study")
    report = resolve_selection_file(manifest["source_report"])
    if not report.name.startswith("strain_coupling_" + tag + "_"):
        raise ValueError("Report and study identity disagree")
    if hashlib.sha256(report.read_bytes()).hexdigest() != manifest["report_sha256"]:
        raise ValueError("Source report changed since selection; export a new manifest")
    frame = pd.read_csv(report)
    records = []
    for case in manifest["cases"]:
        label = case["label"]
        if not re.fullmatch(r"[A-Za-z0-9_-]+", label):
            raise ValueError("Unsafe custom label")
        rows = frame.loc[frame.run_id.eq(case["run_id"])]
        if len(rows) != 1 or rows.iloc[0].status != "complete":
            raise ValueError(f"{label}: not a unique completed run in the selected report")
        row = rows.iloc[0].copy()
        folder = resolve_folder(ROOT / case["run_folder"])
        if folder.resolve() != resolve_folder(row.run_folder).resolve():
            raise ValueError(f"{label}: manifest/report run folders differ")
        metadata = json.loads((folder / "parameters.json").read_text(encoding="utf-8"))
        keys = ["epsilon0", "gamma", "zeta_eff", "E", "nu", "xi", "A", "K", "Gamma",
                "T", "dt", "t_end", "save_every", "lambda_coupling", "L", "d0"]
        for key in keys:
            if not np.isclose(case["parameters"][key], row[key], atol=1e-12, rtol=1e-12) or not np.isclose(metadata[key], row[key], atol=1e-12, rtol=1e-12):
                raise ValueError(f"{label}: inconsistent {key}")
        if metadata["A"] != 1 or metadata["lambda_coupling"] != 0:
            raise ValueError("This selection must use A=1 and lambda_c=0")
        marker = folder / "sweep_status.json"
        if marker.exists() and json.loads(marker.read_text())["status"] != "finished":
            raise ValueError(f"{label}: saved execution did not finish")
        row["label"] = label
        row["run_folder"] = str(folder)
        records.append(row)
    if not records or len({r['label'] for r in records}) != len(records) or len({r['run_id'] for r in records}) != len(records):
        raise ValueError("Selection must contain unique IDs and labels")
    return pd.DataFrame(records), tag, path

if SELECTION_MODE == "custom":
    candidates, SELECTION_STUDY_TAG, manifest_path = load_custom_selection(SELECTION_MANIFEST)
    CANDIDATE_CSV = manifest_path
    SELECTED_LABELS = candidates.label.tolist()
    EXPECTED_CASES = {r.label: (int(r.run_id), r.epsilon0, r.gamma, r.zeta_eff) for _, r in candidates.iterrows()}
    OUTPUT_DIR = OUTPUT_BASE / SELECTION_STUDY_TAG / manifest_path.stem / EXPORT_STAMP
    COMPARISONS = {}  # Add an explicit custom comparison if desired; no automatic A-H groups.
    DIAGNOSTIC_CASES = []
elif SELECTION_MODE == "report":
    if SELECTION_REPORT is None:
        reports = sorted((ROOT / "outputs/strain_coupling/analysis").glob("strain_coupling_" + REPORT_STUDY_TAG + "_*.csv"))
        if not reports:
            raise FileNotFoundError("No report for this activity scan yet. Complete the optional scan in the phase notebook first.")
        report_path = reports[-1]
    else:
        report_path = resolve_selection_file(SELECTION_REPORT)
    if not report_path.name.startswith("strain_coupling_" + REPORT_STUDY_TAG + "_"):
        raise ValueError("Report does not belong to REPORT_STUDY_TAG")
    report_frame = pd.read_csv(report_path)
    records = []
    for label, (run_id, epsilon0, gamma, zeta_eff) in REPORT_CASES.items():
        import re
        if not re.fullmatch(r"[A-Za-z0-9_-]+", label):
            raise ValueError("Use a simple filename-safe case label")
        source = (pd.read_csv(PASSIVE_REPORT) if VISUALIZATION_PRESET == "local_directors" and label == "Passive"
                  else report_frame)
        rows = source.loc[source.run_id.eq(run_id)]
        if len(rows) != 1 or rows.iloc[0].status != "complete":
            raise ValueError(f"{label}: expected one completed report-local run")
        row = rows.iloc[0].copy()
        if not np.allclose([row.epsilon0,row.gamma,row.zeta_eff], [epsilon0,gamma,zeta_eff], atol=1e-12,rtol=0):
            raise ValueError(f"{label}: unexpected control parameters")
        if row.A != 1 or row.lambda_coupling != 0:
            raise ValueError("Expected A=1 and lambda_coupling=0")
        folder = resolve_folder(row.run_folder)
        marker = folder / "sweep_status.json"
        if marker.exists() and json.loads(marker.read_text())["status"] != "finished":
            raise ValueError(f"{label}: execution did not finish")
        row["label"] = label
        records.append(row)
    if not records:
        raise ValueError("REPORT_CASES is empty")
    candidates = pd.DataFrame(records)
    SELECTED_LABELS = candidates.label.tolist()
    EXPECTED_CASES = dict(REPORT_CASES)
    SELECTION_STUDY_TAG = REPORT_STUDY_TAG
    CANDIDATE_CSV = report_path
    OUTPUT_DIR = OUTPUT_BASE / SELECTION_STUDY_TAG / EXPORT_STAMP
    COMPARISONS = {}
    DIAGNOSTIC_CASES = []
elif SELECTION_MODE == "original":
    if CANDIDATE_CSV is None:
        csvs = sorted((ROOT / "outputs/strain_coupling/figures/sweeps/grid_tend5p0").rglob("verified_candidates.csv"))
        if not csvs:
            raise FileNotFoundError("No verified candidate CSV found; select CANDIDATE_CSV explicitly.")
        CANDIDATE_CSV = csvs[-1]
    else:
        CANDIDATE_CSV = Path(CANDIDATE_CSV)
    candidates = pd.read_csv(CANDIDATE_CSV)
else:
    raise ValueError("SELECTION_MODE must be original, custom or report")
runs = {}
for label in SELECTED_LABELS:
    expected = EXPECTED_CASES[label]
    rows = candidates.loc[candidates["label"].eq(label)]
    if len(rows) != 1:
        raise ValueError(f"Candidate {label} missing or duplicated")
    row = rows.iloc[0]
    if int(row.run_id) != expected[0]:
        raise ValueError(f"Candidate {label}: wrong run ID")
    folder = resolve_folder(row.run_folder)
    metadata = json.loads((folder / "parameters.json").read_text(encoding="utf-8"))
    for key, value in zip(["epsilon0", "gamma", "zeta_eff"], expected[1:]):
        if not np.isclose(row[key], value) or not np.isclose(metadata[key], value):
            raise ValueError(f"Candidate {label}: parameter mismatch for {key}")
    for key in ["E", "nu", "xi", "A", "K", "Gamma", "T", "dt", "t_end", "save_every", "lambda_coupling"]:
        if not np.isclose(metadata[key], row[key], atol=1e-12, rtol=1e-12):
            raise ValueError(f"Candidate {label}: metadata mismatch for {key}")
    if metadata["lambda_coupling"] != 0 or not np.isclose(metadata["d0"], 2*metadata["L"]*metadata["epsilon0"]):
        raise ValueError(f"Candidate {label}: inconsistent loading/corotation metadata")
    if row.status != "complete" or (SELECTION_MODE == "original" and label == "H"):
        print(f"EXCLUDED {label} / ID {int(row.run_id)}: {row.status}; {row.get('message', '')}. No spatial snapshots loaded.")
        continue
    files = sorted((folder / "snapshots").glob("step_*.npz"))
    times = []
    for file in files:
        with np.load(file, allow_pickle=False) as saved:
            times.append(float(saved["t"]))
    times = np.asarray(times)
    if not len(times) or not np.isfinite(times).all() or np.any(np.diff(times) <= 0):
        raise ValueError(f"Invalid snapshot times: {folder}")
    nsteps = int(round(metadata["t_end"] / metadata["dt"]))
    stride = int(metadata["save_every"])
    expected_steps = sorted({0, nsteps, *range(stride, nsteps+1, stride)})
    if [f.name for f in files] != [f"step_{step:05d}.npz" for step in expected_steps] or not np.allclose(times, np.array(expected_steps)*metadata["dt"], atol=1e-8, rtol=0):
        raise ValueError(f"{label}: incomplete or incorrectly timed saved history")
    with np.load(files[0], allow_pickle=False) as first:
        coords = first["coords"].copy()
        triangles = first["connectivity"].copy()
    if coords.ndim != 2 or coords.shape[1] != 2 or not np.isfinite(coords).all():
        raise ValueError("Invalid reference coordinates")
    if triangles.ndim != 2 or triangles.shape[1] != 3 or not np.issubdtype(triangles.dtype, np.integer) or triangles.min()<0 or triangles.max()>=len(coords):
        raise ValueError("Connectivity must index saved reference coordinates")
    edges = coords[triangles[:,1:]]-coords[triangles[:,:1]]
    if np.any(np.abs(np.linalg.det(edges)) < 1e-14):
        raise ValueError("Degenerate reference triangle")
    all_edges = np.sort(np.concatenate([triangles[:,[0,1]],triangles[:,[1,2]],triangles[:,[2,0]]]),axis=1)
    unique, counts = np.unique(all_edges,axis=0,return_counts=True)
    runs[label] = dict(label=label,row=row,metadata=metadata,folder=folder,files=files,times=times,
                       coords=coords,triangles=triangles,boundary=unique[counts==1],cache={})
print("Verified candidate source:", CANDIDATE_CSV)
print("Completed candidates available:", ", ".join(runs))


Verified candidate source: /mnt/c/3.Programming Apps/python of the gauds/projects/Master thesis/FEniCS code/outputs/strain_coupling/analysis/strain_coupling_refined_F_tend5p0_20261103T120501_488511Z.csv
Completed candidates available: F_021, F_022, F_023, Passive


## Saved displacement and local strain

The deformed outline uses actual saved displacement; magnification is explicitly labelled. Local gradients are reconstructed from P1 displacement and do not change when the display scale changes. Inverted configurations are only shown when `ALLOW_INVERTED_PREVIEW=True`, with explicit warnings. Printed inverted fractions count triangles (not area fractions).


In [23]:
def load_snapshot(run, target):
    index = int(np.argmin(np.abs(run["times"]-target)))
    actual = float(run["times"][index])
    if not np.isclose(actual,target,atol=1e-8,rtol=0):
        warnings.warn(f"{run['label']}: requested t={target:g}; using saved t={actual:g}")
    if index not in run["cache"]:
        with np.load(run["files"][index],allow_pickle=False) as saved:
            snap = {k:saved[k].copy() for k in ["coords","u","von_mises","Qxx","Qxy","theta","S"]}
        if not all(np.isfinite(v).all() for v in snap.values()):
            raise ValueError(f"{run['label']}: nonfinite frame at t={actual}; no plot produced")
        if not np.array_equal(snap["coords"],run["coords"]) or snap["u"].shape != run["coords"].shape:
            raise ValueError("Snapshot coordinates/displacement shape mismatch")
        if any(snap[k].shape != (len(run["coords"]),) for k in ["S","Qxx","Qxy","theta","von_mises"]):
            raise ValueError("Snapshot scalar shape mismatch")
        if np.any(snap["S"]<0) or np.any(snap["von_mises"]<0):
            raise ValueError("Negative saved scalar magnitude")
        snap["t"]=actual
        run["cache"][index]=snap
    return run["cache"][index]

def deformed_coords(run,snap,scale=1.0):
    return run["coords"] + scale*snap["u"]

def reconstructed_fields(run, snap):
    """Cell gradients of the saved P1 displacement, in reference coordinates."""
    tri = run["triangles"]
    x, u = run["coords"], snap["u"]
    edges = x[tri[:, 1:]] - x[tri[:, :1]]
    du = u[tri[:, 1:]] - u[tri[:, :1]]
    g = np.linalg.solve(edges, du).transpose(0, 2, 1)
    eps = 0.5 * (g + g.transpose(0, 2, 1))
    p = run["metadata"]
    mu = p["E"] / (2 * (1 + p["nu"]))
    return dict(grad_u=g, eps_xy=eps[:, 0, 1],
                eps_norm=np.linalg.norm(eps, axis=(1, 2)),
                detF=np.linalg.det(np.eye(2) + g),
                sigma_xy=2*mu*eps[:, 0, 1] - p["zeta_eff"]*snap["Qxy"][tri].mean(axis=1))


def geometry_diagnostics(run, snap, scale=1.0):
    g = scale * reconstructed_fields(run, snap)["grad_u"]
    eps = 0.5 * (g + g.transpose(0, 2, 1))
    det = np.linalg.det(np.eye(2) + g)
    return dict(strain=float(np.linalg.norm(eps, axis=(1, 2)).max()),
                gradient=float(np.linalg.norm(g, axis=(1, 2)).max()),
                det=float(det.min()), inverted_fraction=float(np.mean(det <= 0)))


def geometry_warning(d):
    if d["det"] <= 0:
        return "NONPHYSICAL PREVIEW\nlocal P1 inversion"
    if d["strain"] > DEFORM_STRAIN_LIMIT or d["gradient"] > DEFORM_GRAD_LIMIT:
        return "Local P1 large-strain/gradient flag"
    return "No local P1 deformation-screen flag"


def coordinates_for_plot(run, snap):
    physical = geometry_diagnostics(run, snap)
    if GEOMETRY == "reference":
        return run["coords"], physical
    if GEOMETRY != "deformed" or not np.isfinite(DEFORM_SCALE) or DEFORM_SCALE <= 0:
        raise ValueError("Choose reference/deformed and a positive finite display scale")
    displayed = geometry_diagnostics(run, snap, DEFORM_SCALE)
    if min(physical["det"], displayed["det"]) <= 0 and not ALLOW_INVERTED_PREVIEW:
        return None, physical
    return deformed_coords(run, snap, DEFORM_SCALE), physical

# Adapted from postprocess.ipynb: average Q components in spatial bins, never angles.
def binned_directors(run,snap,xy,nbins_x=NBINS_X,nbins_y=NBINS_Y):
    ref=run["coords"]
    xedges=np.linspace(ref[:,0].min(),ref[:,0].max(),nbins_x+1)
    yedges=np.linspace(ref[:,1].min(),ref[:,1].max(),nbins_y+1)
    ix=np.clip(np.digitize(ref[:,0],xedges)-1,0,nbins_x-1)
    iy=np.clip(np.digitize(ref[:,1],yedges)-1,0,nbins_y-1)
    bins=ix*nbins_y+iy; count=np.bincount(bins,minlength=nbins_x*nbins_y)
    occupied=count>0
    def average(values):return np.bincount(bins,weights=values,minlength=len(count))[occupied]/count[occupied]
    qxx,qxy=average(snap["Qxx"]),average(snap["Qxy"])
    order=np.hypot(qxx,qxy)
    minimum=np.full(len(count),np.inf);np.minimum.at(minimum,bins,snap["S"])
    # Conservative suppression: no glyph in bins containing any low-S node.
    keep=(minimum[occupied]>=S_GLYPH_MIN)&(order>=S_GLYPH_MIN)
    return average(xy[:,0])[keep],average(xy[:,1])[keep],(.5*np.arctan2(qxy,qxx))[keep]

def director_segments(run,snap,xy):
    xc,yc,theta=binned_directors(run,snap,xy)
    full_length=.65*min(np.ptp(run["coords"][:,0])/NBINS_X,np.ptp(run["coords"][:,1])/NBINS_Y)
    offset=.5*full_length*np.column_stack([np.cos(theta),np.sin(theta)])
    centers=np.column_stack([xc,yc])
    return np.stack([centers-offset,centers+offset],axis=1)

def selected_frames(run):
    targets=list(REQUESTED_TIMES)
    tmax=run["row"].get("time_max_mean_abs_dtheta",np.nan)
    if INCLUDE_MAX_RESPONSE_TIME and np.isfinite(tmax):targets.append(float(tmax))
    frames={}
    for target in targets:
        snap=load_snapshot(run,target)
        frames[snap["t"]]=snap
    return [frames[t] for t in sorted(frames)]


def inspect_failed_case():
    """Read-only checks; finite saved frame does not imply solver convergence/validity."""
    row = candidates.loc[candidates["label"].eq("H")].iloc[0]
    folder = resolve_folder(row.run_folder)
    paths = sorted((folder / "snapshots").glob("step_*.npz"))
    valid, invalid = [], []
    reference = topology = None
    for path in paths:
        try:
            with np.load(path, allow_pickle=False) as z:
                required = ("coords", "u", "Qxx", "Qxy", "S", "theta", "von_mises", "t")
                if any(k not in z or not np.isfinite(z[k]).all() for k in required):
                    raise ValueError("missing/nonfinite fields")
                x = z["coords"]
                if x.ndim != 2 or x.shape[1] != 2 or z["u"].shape != x.shape:
                    raise ValueError("invalid coordinates/displacement shape")
                if any(z[k].shape != (len(x),) for k in required[2:7]):
                    raise ValueError("invalid scalar shape")
                if reference is None:
                    tri = z["connectivity"]
                    if tri.ndim != 2 or tri.shape[1] != 3 or not np.issubdtype(tri.dtype, np.integer):
                        raise ValueError("invalid topology")
                    if tri.min() < 0 or tri.max() >= len(x):
                        raise ValueError("invalid topology indices")
                    if np.any(np.abs(np.linalg.det(x[tri[:,1:]]-x[tri[:,:1]])) < 1e-14):
                        raise ValueError("degenerate reference mesh")
                    reference, topology = x.copy(), tri.copy()
                elif not np.array_equal(x, reference):
                    raise ValueError("reference coordinates changed")
                if "connectivity" in z and not np.array_equal(z["connectivity"], topology):
                    raise ValueError("topology changed")
                if np.any(z["S"] < 0) or np.any(z["von_mises"] < 0):
                    raise ValueError("negative magnitude")
                if not np.allclose(z["S"], np.hypot(z["Qxx"], z["Qxy"]), rtol=1e-7, atol=1e-10):
                    raise ValueError("inconsistent S and Q")
                valid.append((float(z["t"]), path.name))
        except (ValueError, KeyError, OSError) as error:
            invalid.append(f"{path.name}: {error}")
    last = max(valid) if valid else None
    message = f"Recorded status: {row.status}. {row.get('message', '')}"
    message += f"\nChecked {len(paths)} saved files: {len(valid)} finite/structurally consistent; {len(invalid)} rejected."
    message += (f"\nLast finite saved frame: {last[1]}, t={last[0]:g}." if last else
                "\nNo reliable finite saved frame found.")
    if invalid:
        message += "\nFirst rejected file: " + invalid[0]
    message += "\nFinite-frame checks establish saved-data usability only, not P2 convergence or physical admissibility. No H spatial figure is produced."
    return message


## Configurable static fields

`plot_selected` accepts S/directors, displacement magnitude or components, saved total von Mises, P1 shear strain/strain norm, approximate total shear stress, and signed nematic director change. Colours are cell-averaged for display; actual input extrema are annotated. Director change is wrapped to the minimum nematic angle and masked if either initial or current S is low.

Shared scales apply across rows at the requested time. Individual scales are explicitly labelled and are for seeing patterns, not comparing colour intensities. Signed fields use symmetric diverging scales. Log mechanical scales show below-floor values in gray; S is never silently capped. All panels in each figure share coordinate bounds and equal spatial aspect ratio.


In [24]:
FIELD_INFO = {
    "S": ("S + directors", "cividis", False),
    "umag": ("Displacement magnitude |u|", "viridis", False),
    "ux": ("Horizontal displacement u_x", "RdBu_r", True),
    "uy": ("Vertical displacement u_y", "RdBu_r", True),
    "von_mises": ("Saved von Mises: TOTAL stress", "inferno", False),
    "eps_xy": ("Shear strain epsilon_xy: P1 approximation", "RdBu_r", True),
    "eps_norm": ("Strain norm: P1 approximation", "viridis", False),
    "sigma_xy": ("Total shear stress sigma_xy: P1 approximation", "RdBu_r", True),
    "dtheta": ("Signed director change [degrees]", "RdBu_r", True),
}


def field_values(run, snap, field):
    if field in {"S", "von_mises"}:
        return snap[field], False
    if field == "umag":
        return np.linalg.norm(snap["u"], axis=1), False
    if field in {"ux", "uy"}:
        return snap["u"][:, 0 if field == "ux" else 1], False
    if field in {"eps_xy", "eps_norm", "sigma_xy"}:
        return reconstructed_fields(run, snap)[field], True
    if field == "dtheta":
        initial = load_snapshot(run, 0.0)
        theta = .5*np.arctan2(snap["Qxy"], snap["Qxx"])
        theta0 = .5*np.arctan2(initial["Qxy"], initial["Qxx"])
        delta = .5*np.arctan2(np.sin(2*(theta-theta0)), np.cos(2*(theta-theta0)))
        valid = (snap["S"] >= S_GLYPH_MIN) & (initial["S"] >= S_GLYPH_MIN)
        return np.where(valid, np.degrees(delta), np.nan), False
    raise ValueError(field)


def make_norm(arrays, signed=False, logarithmic=False):
    values = np.concatenate([a[np.isfinite(a)] for a in arrays])
    if signed:
        limit = max(float(np.abs(values).max()), 1e-12) if len(values) else 1.0
        return Normalize(-limit, limit)
    vmax = max(float(values.max()), 1e-12) if len(values) else 1.0
    return LogNorm(vmax*1e-4, vmax) if logarithmic else Normalize(0, vmax)


produced = []


def plot_selected(labels, target=2.5, fields=("S", "umag", "von_mises"),
                  scale_mode="shared", log_fields=(), vectors=False, name="comparison", entries=None):
    """Static saved fields; shared coordinates and optional shared colour limits."""
    if scale_mode not in {"shared", "individual"}:
        raise ValueError("Choose shared or individual scales")
    if any(f not in FIELD_INFO for f in fields):
        raise ValueError("Unknown field")
    if any(FIELD_INFO[f][2] for f in log_fields):
        raise ValueError("Signed fields require linear diverging scales")
    if entries is None:
        entries = [(runs[label], load_snapshot(runs[label], target)) for label in labels]
    positions = [coordinates_for_plot(r, s) for r, s in entries]
    values = [[field_values(r, s, f) for f in fields] for r, s in entries]
    bounds = np.vstack([r["coords"] for r, s in entries] +
                       [xy for xy, d in positions if xy is not None])
    low, high = bounds.min(axis=0), bounds.max(axis=0)
    pad = .04*np.maximum(high-low, .1)
    norms = {f: make_norm([row[j][0] for row in values], FIELD_INFO[f][2], f in log_fields)
             for j, f in enumerate(fields)}
    fig, axes = plt.subplots(len(entries), len(fields), squeeze=False,
                             figsize=(4.2*len(fields), 4.8*len(entries)+1), layout="constrained")
    for row, ((run, snap), (xy, d)) in enumerate(zip(entries, positions)):
        p = run["metadata"]
        status = geometry_warning(d)
        if GEOMETRY == "deformed" and geometry_diagnostics(run, snap, DEFORM_SCALE)["det"] <= 0:
            status = "NONPHYSICAL PREVIEW\nlocal P1 inversion"
        smax = float(snap["S"].max())
        if smax > 1 + S_WARNING_TOL:
            status += f"\nSmax - 1 = {smax - 1:.2e}"
            if smax > S_EXTREME_LEVEL:
                status += " (high order)"
        low_fraction = float(np.mean(snap["S"] < S_GLYPH_MIN))
        if low_fraction:
            status += f"\nLow S: {100*low_fraction:.1f}% of nodes"
        print(f"{run['label']} t={snap['t']:g}: min det(F)={d['det']:.5g}, "
              f"max ||epsilon||={d['strain']:.5g}, max ||grad u||={d['gradient']:.5g}, "
              f"inverted triangles={100*d['inverted_fraction']:.2f}%")
        for col, field in enumerate(fields):
            ax = axes[row, col]
            title, cmap_name, signed = FIELD_INFO[field]
            v, cell_based = values[row][col]
            norm = norms[field] if scale_mode == "shared" else make_norm([v], signed, field in log_fields)
            ax.set_title(f"{run['label']} / ID {int(run['row'].run_id)} | t={snap['t']:g}\n{title}", fontsize=10, pad=58)
            if xy is None:
                ax.text(.5, .5, "Inverted preview disabled", transform=ax.transAxes, ha="center")
                ax.set_axis_off()
                continue
            tri = run["triangles"]
            cmap = plt.get_cmap(cmap_name).copy()
            cmap.set_bad("#dddddd"); cmap.set_under("#dddddd")
            # Explicit polygons also permit labelled folded mathematical previews.
            # Any cell with undefined director change remains masked.
            cell_v = v if cell_based else v[tri].mean(axis=1)
            display_v = np.ma.masked_invalid(cell_v)
            if isinstance(norm, LogNorm):
                display_v = np.maximum(display_v, np.nextafter(norm.vmin, 0))
            artist = PolyCollection(xy[tri], array=display_v, cmap=cmap, norm=norm,
                                    edgecolors="none", rasterized=True)
            ax.add_collection(artist)
            ax.add_collection(LineCollection(xy[run["boundary"]], colors="black", linewidths=.9))
            if GEOMETRY == "deformed":
                ax.add_collection(LineCollection(run["coords"][run["boundary"]], colors="gray",
                                                 linewidths=.8, linestyles="dashed"))
            if field == "S":
                glyphs = LineCollection(director_segments(run, snap, xy), colors="crimson", linewidths=1.1)
                glyphs.set_path_effects([pe.Stroke(linewidth=1.9, foreground="white"), pe.Normal()])
                ax.add_collection(glyphs)
            if vectors and field in {"umag", "ux", "uy"}:
                ids = np.arange(0, len(xy), max(1, len(xy)//60))
                ax.quiver(run["coords"][ids, 0], run["coords"][ids, 1],
                          snap["u"][ids, 0], snap["u"][ids, 1], angles="xy",
                          scale_units="xy", scale=1, color="black", width=.003)
            ax.set_xlim(low[0]-pad[0], high[0]+pad[0]); ax.set_ylim(low[1]-pad[1], high[1]+pad[1])
            ax.set_aspect("equal"); ax.set_xlabel("x"); ax.set_ylabel("y")
            ax.text(.5, 1.01, status, transform=ax.transAxes, va="bottom", ha="center", fontsize=8, color="darkred",
                    bbox=dict(facecolor="white", alpha=.85, edgecolor="none"), wrap=True)
            finite = v[np.isfinite(v)]
            if len(finite):
                ax.text(.02, .02, f"data range {finite.min():.3g} to {finite.max():.3g}",
                        transform=ax.transAxes, fontsize=7,
                        bbox=dict(facecolor="white", alpha=.85, edgecolor="none"))
            cb = fig.colorbar(artist, ax=ax, location="bottom", pad=.04,
                              extend="min" if isinstance(norm, LogNorm) else "neither")
            cb.set_label(scale_mode + " scale" +
                         ("; log, gray below floor" if isinstance(norm, LogNorm) else ""), fontsize=8)
            cb.ax.tick_params(labelsize=8)
        axes[row, 0].set_ylabel(f"epsilon0={p['epsilon0']:g}, gamma={p['gamma']:g}, "
                               f"zeta_eff={p['zeta_eff']:g}\ny", fontsize=9)
    geometry_title = "Reference geometry" if GEOMETRY == "reference" else (
        f"Saved deformation: X + {DEFORM_SCALE:g} u; dashed = reference")
    fig.suptitle(" / ".join(dict.fromkeys(labels)) + "\n" + geometry_title, fontsize=13)
    footer = ("Saved P1 displacement; cell-averaged scalar display. Q directors have no extra rotation; low-S glyphs hidden.\n"
              "Inverted previews are nonphysical; overlapping triangles depend on draw order.")
    if vectors:
        footer += "\nDisplacement arrows: true u, scale 1, starting at reference coordinates."
    fig.supxlabel(footer, fontsize=8)
    if SAVE_PDFS:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        suffix = "" if "timepoints" in name else f"_t{target:g}"
        path = OUTPUT_DIR / f"{name}_{GEOMETRY}_{scale_mode}{suffix}.pdf"
        if path.exists() and not OVERWRITE_PDFS:
            plt.close(fig)
            raise FileExistsError(f"{path.name} exists: change name or rerun configuration for a new folder")
        # Replace this export only after a complete PDF has been written.
        temporary = path.with_name(path.stem + ".pending.pdf")
        fig.savefig(temporary, dpi=300, bbox_inches="tight")
        temporary.replace(path)
        produced.append(path)
    plt.show(); plt.close(fig)


## Optional static snapshots

Skipped unless `GENERATE_STATIC_SNAPSHOTS=True` in the first configuration cell. With `INCLUDE_MECHANICAL_FIELDS=False`, selected snapshots contain only S/directors. Set both switches True to restore the prior full-field/static workflow, including separate mechanical diagnostics. No plotting functions were removed.


In [25]:
if GENERATE_STATIC_SNAPSHOTS:
    static_fields = ("S", "umag", "von_mises") if INCLUDE_MECHANICAL_FIELDS else ("S",)
    # A-G: one three-field figure per case, with deduplicated time rows.
    for label, run in runs.items():
        entries = [(run, snap) for snap in selected_frames(run)]
        plot_selected([label], entries=entries, fields=static_fields,
                      name=f"case_{label}_ID{int(run['row'].run_id):03d}_timepoints")

    # Peak comparisons: shared quantitative scales plus individual detail scales.
    # The metadata labels must be retained: DG and EF change multiple parameters.
    for group, labels in COMPARISONS.items():
        ids = "_".join(str(int(runs[label]["row"].run_id)) for label in labels)
        for mode in ("shared", "individual"):
            plot_selected(labels, target=COMPARISON_TIME, scale_mode=mode, fields=static_fields,
                          name=f"comparison_{group}_IDs{ids}")

    # Separate mechanical diagnostics at peak displacement; do not crowd main figures.
    if INCLUDE_MECHANICAL_FIELDS:
        for label in DIAGNOSTIC_CASES:
            run_id = int(runs[label]["row"].run_id)
            plot_selected([label], target=COMPARISON_TIME, fields=("ux", "uy"),
                          name=f"diagnostic_{label}_ID{run_id}_displacement")
            plot_selected([label], target=COMPARISON_TIME, fields=("eps_xy", "eps_norm", "sigma_xy"),
                          name=f"diagnostic_{label}_ID{run_id}_strain_shear_stress")

    if SELECTION_MODE == "original":
        # H: report finite saved data separately, never add H to completed-case plots.
        summary = inspect_failed_case()
        print(summary)
        if SAVE_PDFS:
            import textwrap
            fig, ax = plt.subplots(figsize=(10, 5), layout="constrained")
            ax.set_axis_off()
            ax.text(0, 1, "Case H / ID 188: failed-run status", va="top", fontsize=15)
            paragraphs = [textwrap.fill(line, width=95) for line in summary.splitlines()]
            ax.text(0, .88, "\n\n".join(paragraphs), va="top", fontsize=10, transform=ax.transAxes)
            path = OUTPUT_DIR / "case_H_ID188_failure_status.pdf"
            temporary = path.with_name(path.stem + ".pending.pdf")
            fig.savefig(temporary, bbox_inches="tight"); temporary.replace(path)
            produced.append(path)
            plt.show(); plt.close(fig)

    print("PDF output directory:", OUTPUT_DIR)
    print("PDFs produced:", len(produced))
    for path in produced:
        print(path.name)
else:
    print("Static snapshot exports disabled.")


Static snapshot exports disabled.


## Animation configuration and export

The generation switch and director density are in the first configuration cell. GIF is the preferred format; FPS and DPI are below. No FFmpeg is needed unless optional MP4 export is enabled. Custom director groups export only the configured comparisons. New study/manifest/timestamp directories preserve old movies.

Run all cells after setting `GENERATE_ANIMATIONS=True`; static generation remains off by default. GIF validation reopens every frame, checks frame count and chronological rendered-panel correspondence, and checks that visible field changes survive encoding. Validation results remain in memory; no JSON sidecar is saved. Failures raise an error rather than accepting the output.


In [26]:
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter, writers
import matplotlib as mpl
import subprocess
import imageio_ffmpeg

ANIMATIONS_TO_EXPORT = ["A","BC", "E", "F", "DG"]
ANIMATION_FPS = 10
ANIMATION_DPI = 100  # Compact email GIF; increase for presentation resolution.
EXPORT_MP4 = True
EXPORT_GIF = VISUALIZATION_PRESET != "local_directors"  # Uses Pillow; FFmpeg is not needed.
FFMPEG_PATH = imageio_ffmpeg.get_ffmpeg_exe()  # GIF needs Pillow only; set an existing FFmpeg path for optional MP4.
ANIMATION_LOG_FIELDS = {"DG": ("umag", "von_mises")}  # Others default to linear.
ANIMATION_GROUPS = {
    "A": (["A"], "A_baseline_shear"),
    "BC": (["B", "C"], "BC_coupling_sign_comparison"),
    "E": (["E"], "E_activity_driven"),
    "F": (["F"], "F_delayed_nematic_response"),
    "DG": (["D", "G"], "DG_model_limits"),
    "D": (["D"], "D_model_limits"),  # Optional separate extreme-case views.
    "G": (["G"], "G_model_limits"),
}
export_stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%fZ")
study_folder = SELECTION_STUDY_TAG if SELECTION_MODE in ("custom", "report") else "original"
ANIMATION_DIR = (ROOT / "outputs/strain_coupling/figures/sweeps/animations" / study_folder /
                 f"{VISUALIZATION_PRESET}_{export_stamp}")
animation_outputs = []

# Higher-resolution director GIFs: edit bin counts here, then rerun definitions.
ANIMATION_NBINS_X, ANIMATION_NBINS_Y = DIRECTOR_BINS_X, DIRECTOR_BINS_Y
if SELECTION_MODE in ("custom", "report"):
    ANIMATION_GROUPS = {label: ([label], f"{label}_ID{int(runs[label]['row'].run_id)}_director_fields")
                        for label in SELECTED_LABELS}
    ANIMATIONS_TO_EXPORT = list(SELECTED_LABELS)  # Only explicit manifest choices; generation remains off.
    ANIMATION_LOG_FIELDS = {}

# Director-only groups may contain 1, 2 or 3 manifest-selected cases.
if not INCLUDE_MECHANICAL_FIELDS and SELECTION_MODE in ("custom", "report"):
    for group, labels in DIRECTOR_COMPARISONS.items():
        if not 1 <= len(labels) <= 3 or len(labels) != len(set(labels)):
            raise ValueError(f"{group}: choose 1-3 distinct cases")
        if any(label not in runs for label in labels):
            raise ValueError(f"{group}: missing cases; select the matching manifest or edit DIRECTOR_COMPARISONS")
        for label in labels:
            if label in DIRECTOR_CASE_CHECKS:
                study, run_id, eps, gamma, zeta = DIRECTOR_CASE_CHECKS[label]
                row = runs[label]["row"]
                if SELECTION_STUDY_TAG != f"refined_{study}_tend5p0" or int(row.run_id) != run_id:
                    raise ValueError(f"{label}: unexpected study/ID")
                if not np.allclose([row.epsilon0,row.gamma,row.zeta_eff], [eps,gamma,zeta], atol=1e-12, rtol=0):
                    raise ValueError(f"{label}: unexpected control parameters")
    ANIMATION_GROUPS = {key:(labels,key+"_directors") for key,labels in DIRECTOR_COMPARISONS.items()}
    ANIMATIONS_TO_EXPORT = list(DIRECTOR_COMPARISONS)
    ANIMATION_LOG_FIELDS = {}  # Shared linear S scale across all panels/frames.


In [27]:
def prepare_movie(labels, log_fields=(), fields=("S", "umag", "von_mises")):
    data, schedule = {}, []
    for label in labels:
        if (SELECTION_MODE == "original" and label == "H") or label not in runs:
            raise ValueError(f"{label}: not a completed loaded candidate")
        run = runs[label]
        data[label] = {}
        for t in run["times"]:
            # Match round-off only, not different simulation times.
            matches = [v for v in schedule if abs(v-float(t)) <= 1e-8]
            key = matches[0] if matches else float(t)
            if not matches:
                schedule.append(key)
            if key in data[label]:
                raise ValueError(f"Duplicate timestamp for {label}: {t}")
            try:
                snap = load_snapshot(run, float(t))
                if abs(snap["t"]-float(t)) > 1e-8:
                    raise ValueError("Timestamp mismatch")
                xy = deformed_coords(run, snap, scale=1.0)
                diagnostic = geometry_diagnostics(run, snap, scale=1.0)
                data[label][key] = (snap, xy, diagnostic)
            except (ValueError, OSError) as error:
                warnings.warn(f"{label} t={t:g}: frame rejected: {error}")
    schedule = sorted(schedule)
    usable = [item for frames in data.values() for item in frames.values()]
    if not usable:
        raise ValueError("No finite frames available")
    for label in labels:
        absent = [t for t in schedule if t not in data[label]]
        if absent and VISUALIZATION_PRESET == "local_directors":
            raise ValueError(f"{label}: missing or rejected saved frames: {absent}")
        if absent:
            print(f"{label}: missing/rejected frames, explicitly blank: {absent}")
        print(f"{label}: {len(data[label])} finite saved frames")
    print(f"Timeline: {len(schedule)} timestamps, {schedule[0]:g} to {schedule[-1]:g}; no interpolation")
    for endpoint in (0, 2.5, 5):
        if not any(abs(t-endpoint) <= 1e-8 for t in schedule):
            warnings.warn(f"Requested landmark t={endpoint:g} is absent from saved movie times")
    bounds = np.vstack([runs[l]["coords"] for l in labels] + [v[1] for v in usable])
    low, high = bounds.min(axis=0), bounds.max(axis=0)
    pad = .04*np.maximum(high-low, .1)
    norms = {f: make_norm([field_values(runs[l], item[0], f)[0]
                          for l in labels for item in data[l].values()], False, f in log_fields)
             for f in fields}
    return fields, data, schedule, norms, low-pad, high+pad


def animation_director_segments(run, snap, xy):
    nx, ny = ANIMATION_NBINS_X, ANIMATION_NBINS_Y
    if not isinstance(nx, int) or not isinstance(ny, int) or nx < 1 or ny < 1:
        raise ValueError("Director bin counts must be positive integers")
    xc, yc, theta = binned_directors(run, snap, xy, nbins_x=nx, nbins_y=ny)
    length = .65*min(np.ptp(run["coords"][:,0])/nx, np.ptp(run["coords"][:,1])/ny)
    offset = .5*length*np.column_stack([np.cos(theta), np.sin(theta)])
    centers = np.column_stack([xc, yc])
    return np.stack([centers-offset, centers+offset], axis=1)


def build_full_field_movie(labels, log_fields=()):
    fields, data, times, norms, low, high = prepare_movie(labels, log_fields)
    fig, axes = plt.subplots(len(labels), 3, squeeze=False,
                             figsize=(13, 5*len(labels)+1.4))
    fig.subplots_adjust(left=.08, right=.96, bottom=.15, top=.83, hspace=.85, wspace=.42)
    artists = {}
    header = fig.suptitle("", fontsize=12)
    fig.text(.5, .02, "Saved P1 displacement; cell-averaged colours; Q directors without extra rotation.\n"
             "Inverted shapes are nonphysical previews; overlap colours depend on drawing order. Fixed saved-frame playback FPS.",
             ha="center", fontsize=8)
    for row, label in enumerate(labels):
        run = runs[label]; p = run["metadata"]
        artists[label] = []
        for col, field in enumerate(fields):
            ax = axes[row, col]
            ax.set_xlim(low[0], high[0]); ax.set_ylim(low[1], high[1]); ax.set_aspect("equal")
            ax.set_xlabel("x"); ax.set_ylabel("y")
            ax.set_autoscale_on(False)
            ax.set_title(FIELD_INFO[field][0], fontsize=10)
            cmap = plt.get_cmap(FIELD_INFO[field][1]).copy()
            cmap.set_bad("#dedede"); cmap.set_under("#dedede")
            mesh = PolyCollection([], cmap=cmap, norm=norms[field], edgecolors="none")
            outline = LineCollection([], colors="black", linewidths=.9)
            glyph = LineCollection([], colors="crimson", linewidths=1.1)
            glyph.set_path_effects([pe.Stroke(linewidth=1.9, foreground="white"), pe.Normal()])
            for artist in (mesh, outline, glyph): ax.add_collection(artist)
            ax.add_collection(LineCollection(run["coords"][run["boundary"]], colors="gray",
                                             linestyles="dashed", linewidths=.8))
            missing = ax.text(.5, .5, "", transform=ax.transAxes, ha="center", color="darkred")
            if row == 0:
                # One permanent colorbar per field for all rows; never recreated.
                cax = fig.add_axes([.10 + col*.30, .075, .23, .018])
                cb = fig.colorbar(mesh, cax=cax, orientation="horizontal",
                                 extend="min" if isinstance(norms[field], LogNorm) else "neither")
                cb.set_label("Shared, fixed scale" + ("; log / gray below floor" if field in log_fields else ""), fontsize=8)
                cb.ax.tick_params(labelsize=8)
            artists[label].append((mesh, outline, glyph, missing))
        # Row warnings occupy reserved space above the panels, not the data.
        y = .94 if len(labels)==1 else (.94 if row==0 else .51)
        artists[label+"_text"] = fig.text(.5, y, "", ha="center", va="top", fontsize=9)

    def update(index):
        t = times[index]
        header.set_text(f"Saved deformation X + u (scale 1); dashed = reference | timeline t={t:g}")
        for label in labels:
            run = runs[label]; p = run["metadata"]
            item = data[label].get(t)
            prefix = (f"{label} / ID {int(run['row'].run_id)} | epsilon0={p['epsilon0']:g}, "
                      f"gamma={p['gamma']:g}, zeta_eff={p['zeta_eff']:g}")
            if item is None:
                artists[label+"_text"].set_text(prefix + "\nMISSING / REJECTED saved frame")
                for mesh, outline, glyph, missing in artists[label]:
                    mesh.set_verts([]); mesh.set_array(np.array([])); outline.set_segments([]); glyph.set_segments([])
                    missing.set_text("No finite saved state at this time")
                continue
            snap, xy, diagnostic = item
            actual = snap["t"]
            if p["epsilon0"] == 0:
                stage = "no imposed boundary actuation"
            else:
                peak = p["T"]/2
                stage = "peak actuation" if abs(actual-peak)<1e-8 else "loading" if actual<peak else "unloading" if actual<=p["T"] else "post-actuation"
                displacement = p["d0"]*np.sin(np.pi*actual/p["T"]) if 0<=actual<=p["T"] else 0
                stage += f"; d(t)={displacement:.4g}"
            warning = geometry_warning(diagnostic).replace("\n", ": ")
            smax = float(snap["S"].max()); fraction = float(np.mean(snap["S"]<S_GLYPH_MIN))
            if fraction: warning += f"; low S: {100*fraction:.1f}% nodes"
            if smax>1+S_WARNING_TOL:
                warning += f"; Smax-1={smax-1:.2e}" + (" (high order)" if smax>S_EXTREME_LEVEL else "")
            artists[label+"_text"].set_text(prefix + f" | saved t={actual:g}\n" + stage + "\n" + warning)
            artists[label+"_text"].set_color("darkred" if diagnostic["det"]<=0 else "black")
            for col, field in enumerate(fields):
                mesh, outline, glyph, missing = artists[label][col]
                v = field_values(run, snap, field)[0][run["triangles"]].mean(axis=1)
                if isinstance(norms[field], LogNorm): v=np.maximum(v,np.nextafter(norms[field].vmin,0))
                mesh.set_verts(xy[run["triangles"]]); mesh.set_array(v)
                outline.set_segments(xy[run["boundary"]]); missing.set_text("")
                glyph.set_segments(animation_director_segments(run,snap,xy) if field=="S" else [])
        return []

    movie = FuncAnimation(fig, update, frames=len(times), interval=1000/ANIMATION_FPS, blit=False, cache_frame_data=False)
    movie.review = dict(times=times, field_axes=list(axes.flat), data=data, labels=labels, norms=norms)
    return fig, movie, len(times)


def low_order_area_fraction(run, snap):
    """Exact area below threshold for the piecewise-linear saved S field."""
    tri = run["triangles"]
    points = run["coords"][tri]
    areas = .5*np.abs(np.linalg.det(points[:,1:]-points[:,:1]))
    a,b,c = np.sort(snap["S"][tri],axis=1).T
    t = S_GLYPH_MIN
    fraction = np.zeros(len(tri))
    fraction[(c < t) | ((c == t) & (a < c))] = 1
    lower = (a < t) & (t < b)
    upper = (b <= t) & (t < c)
    fraction[lower] = (t-a[lower])**2/((b[lower]-a[lower])*(c[lower]-a[lower]))
    fraction[upper] = 1-(c[upper]-t)**2/((c[upper]-a[upper])*(c[upper]-b[upper]))
    return float(np.average(fraction, weights=areas))


def director_warning(run, snap, diagnostic):
    if diagnostic["det"] <= 0:
        lines = ["NONPHYSICAL PREVIEW: P1 inversion"]
    elif diagnostic["strain"] > DEFORM_STRAIN_LIMIT or diagnostic["gradient"] > DEFORM_GRAD_LIMIT:
        lines = ["Local P1 large-strain/gradient flag"]
    else:
        lines = ["No local P1 mechanical flag"]
    fraction = low_order_area_fraction(run, snap)
    lines.append(f"S < {S_GLYPH_MIN:g}: {100*fraction:.1f}% reference area")
    smax = float(snap["S"].max())
    if smax > 1+S_WARNING_TOL:
        lines.append(f"Smax - 1 = {smax-1:.2e}" + (" (high order)" if smax>S_EXTREME_LEVEL else ""))
    return "\n".join(lines)


def build_director_movie(labels):
    if not 1 <= len(labels) <= 3 or len(labels)!=len(set(labels)):
        raise ValueError("Choose 1-3 distinct simulations")
    _, data, times, norms, low, high = prepare_movie(labels, fields=("S",))
    fig, axes = plt.subplots(1, len(labels), squeeze=False, figsize=((8, 9) if VISUALIZATION_PRESET == "local_directors" else (4*len(labels)+.7,6.6)))
    fig.subplots_adjust(left=.07,right=.88,bottom=.25,top=.80,wspace=.35)
    header = fig.suptitle("",fontsize=13,y=.98)
    artists={}; panels=list(axes.flat)
    for ax,label in zip(panels,labels):
        run=runs[label];p=run["metadata"]
        ax.set_xlim(low[0],high[0]);ax.set_ylim(low[1],high[1]);ax.set_aspect("equal");ax.set_autoscale_on(False)
        ax.set_xlabel("x");ax.set_ylabel("y")
        ax.set_title(f"{label} / ID {int(run['row'].run_id)}\n"
                     rf"$\gamma={p['gamma']:g}$, $\zeta_{{eff}}={p['zeta_eff']:g}$, $\varepsilon_0={p['epsilon0']:g}$",
                     fontsize=10,pad=12)
        mesh=PolyCollection([],cmap="cividis",norm=norms["S"],edgecolors="none")
        outline=LineCollection([],colors="black",linewidths=.8)
        glyph=LineCollection([],colors="crimson",linewidths=.85)
        glyph.set_path_effects([pe.Stroke(linewidth=1.5,foreground="white"),pe.Normal()])
        for artist in (mesh,outline,glyph):ax.add_collection(artist)
        ax.add_collection(LineCollection(run["coords"][run["boundary"]],colors="gray",linestyles="dashed",linewidths=.6))
        missing=ax.text(.5,.5,"",transform=ax.transAxes,ha="center",color="darkred")
        box=ax.get_position();center=(box.x0+box.x1)/2
        warning=fig.text(center,.17,"",ha="center",va="top",fontsize=8)
        artists[label]=(mesh,outline,glyph,missing,warning)
    cax=fig.add_axes([.92,.29,.018,.45])
    cb=fig.colorbar(mesh,cax=cax);cb.set_label("Nematic order S (shared, fixed)",fontsize=10)
    fig.text(.5,.025,"X + u, scale 1; dashed = reference. Saved P1 displacement approximates original P2 mechanics.\n"
             "Q directors have no extra rotation; low-S glyphs hidden. Inverted previews are not physical shapes.",
             ha="center",fontsize=8)

    def update(index):
        t=times[index]
        header.set_text(f"Director evolution | saved simulation time t = {t:g}")
        for label in labels:
            run=runs[label];item=data[label].get(t)
            mesh,outline,glyph,missing,warning=artists[label]
            if item is None:
                mesh.set_verts([]);mesh.set_array(np.array([]));outline.set_segments([]);glyph.set_segments([])
                missing.set_text("No saved finite state\nat this timestamp")
                warning.set_text("MISSING / REJECTED frame");warning.set_color("darkred")
                continue
            snap,xy,diagnostic=item
            mesh.set_verts(xy[run["triangles"]]);mesh.set_array(snap["S"][run["triangles"]].mean(axis=1))
            outline.set_segments(xy[run["boundary"]]);glyph.set_segments(animation_director_segments(run,snap,xy))
            missing.set_text("");warning.set_text(director_warning(run,snap,diagnostic))
            warning.set_color("darkred" if diagnostic["det"]<=0 else "black")
        return []

    movie=FuncAnimation(fig,update,frames=len(times),interval=1000/ANIMATION_FPS,blit=False,cache_frame_data=False)
    movie.review=dict(times=times,field_axes=panels,data=data,labels=labels,norms=norms,artists=artists)
    return fig,movie,len(times)


def build_movie(labels, log_fields=()):
    return build_full_field_movie(labels,log_fields) if INCLUDE_MECHANICAL_FIELDS else build_director_movie(labels)


class ReviewedPillowWriter(PillowWriter):
    """Retain small rendered panel samples to validate decoded GIF field changes."""
    def __init__(self, fig, movie, **kwargs):
        super().__init__(**kwargs)
        self.review_fig=fig;self.review=movie.review;self.samples=[]
    def grab_frame(self, **kwargs):
        super().grab_frame(**kwargs)
        frame=self._frames[-1].convert("RGB")
        width,height=frame.size
        boxes=[]
        for ax in self.review["field_axes"]:
            b=ax.get_position()
            boxes.append((int(b.x0*width),int((1-b.y1)*height),int(b.x1*width),int((1-b.y0)*height)))
        self.boxes=boxes
        self.samples.append(np.stack([np.asarray(frame.crop(b).resize((64,64)),dtype=float) for b in boxes]))


def validate_saved_gif(path, writer, count):
    from PIL import Image as PILImage
    times=writer.review["times"]
    if count<2 or len(times)!=count or np.any(np.diff(times)<=0):
        raise ValueError("Expected multiple chronological saved timestamps")
    decoded=[]
    with PILImage.open(path) as gif:
        if gif.n_frames!=count:
            raise ValueError(f"GIF has {gif.n_frames} frames; expected {count}")
        for i in range(count):
            gif.seek(i);frame=gif.convert("RGB");frame.load()
            sample=np.stack([np.asarray(frame.crop(b).resize((64,64)),dtype=float) for b in writer.boxes])
            if np.abs(sample-writer.samples[i]).mean()>12:
                raise ValueError(f"GIF frame {i} does not match its chronological render")
            decoded.append(sample)
    # Compare field pixels, excluding time labels and warnings, rather than relying on a changing clock.
    rendered_change=max(float(np.abs(s-writer.samples[0]).mean()) for s in writer.samples)
    decoded_change=max(float(np.abs(s-decoded[0]).mean()) for s in decoded)
    if rendered_change>.1 and decoded_change<=.05:
        raise ValueError("Rendered field changes were lost in GIF export")
    if rendered_change<=.1:
        warnings.warn("Fields are visually nearly unchanged at this resolution; inspect shared scales/glyph density")
    return dict(frames=count,times=times,fps=ANIMATION_FPS,rendered_panel_change=rendered_change,
                decoded_panel_change=decoded_change,validation="all frames reopened and compared in chronological order")


def export_movie(key):
    if not GENERATE_ANIMATIONS:
        return
    if ANIMATION_FPS <= 0 or ANIMATION_DPI <= 0:
        raise ValueError("FPS and DPI must be positive")
    if not EXPORT_MP4 and not EXPORT_GIF:
        raise ValueError("Select MP4 or GIF export")
    if FFMPEG_PATH:
        mpl.rcParams["animation.ffmpeg_path"] = str(FFMPEG_PATH)
    if EXPORT_MP4 and not writers.is_available("ffmpeg"):
        raise RuntimeError("FFmpeg is unavailable in this kernel. Point FFMPEG_PATH to an existing encoder; no dependency was installed.")
    labels, filename = ANIMATION_GROUPS[key]
    fig, movie, count = build_movie(labels, ANIMATION_LOG_FIELDS.get(key, ()))
    try:
        if EXPORT_MP4:
            path = ANIMATION_DIR / "MP4" / (filename + ".mp4")
            path.parent.mkdir(parents=True, exist_ok=True)
            if path.exists(): raise FileExistsError(path)
            temporary = path.with_name(path.stem + ".pending.mp4")
            writer = FFMpegWriter(fps=ANIMATION_FPS, codec="libx264", extra_args=[
                "-pix_fmt", "yuv420p", "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2", "-movflags", "+faststart"])
            print(f"Exporting {key}: {count} frames, {count/ANIMATION_FPS:.1f} playback seconds")
            movie.save(temporary, writer=writer, dpi=ANIMATION_DPI)
            # Decode every frame; this checks readability, not just file existence.
            check = subprocess.run([FFMpegWriter.bin_path(), "-v", "error", "-xerror", "-i", str(temporary),
                                    "-map", "0:v:0", "-vsync", "0", "-progress", "pipe:1", "-f", "null", "-"], capture_output=True, text=True)
            if check.returncode or check.stderr.strip():
                raise RuntimeError("MP4 decode validation failed: " + check.stderr)
            decoded = [int(line.split("=", 1)[1]) for line in check.stdout.splitlines() if line.startswith("frame=")]
            if not decoded or decoded[-1] != count:
                raise RuntimeError(f"MP4 decoded frame count differs from the {count} scheduled frames")
            temporary.replace(path)
            animation_outputs.append((str(path), count, "MP4 fully decoded; saved timeline frame count verified"))
        if EXPORT_GIF:
            path = ANIMATION_DIR / "GIF" / (filename + ".gif")
            path.parent.mkdir(parents=True, exist_ok=True)
            if path.exists(): raise FileExistsError(path)
            temporary = path.with_name(path.stem + ".pending.gif")
            writer = ReviewedPillowWriter(fig, movie, fps=ANIMATION_FPS)
            print(f"Exporting {key}: {count} chronological saved frames", flush=True)
            movie.save(temporary, writer=writer, dpi=ANIMATION_DPI)
            validation = validate_saved_gif(temporary, writer, count)
            temporary.replace(path)
            # Validation stays in memory; no JSON sidecar is written.
            animation_outputs.append((str(path), count, "GIF reopened; frame count, chronology and field changes checked"))
    finally:
        plt.close(fig)


### Individual-case exports

In [28]:
if GENERATE_ANIMATIONS:
    for key in ANIMATIONS_TO_EXPORT:
        if key not in ANIMATION_GROUPS:
            raise ValueError(f"Unknown animation: {key}")
        if len(ANIMATION_GROUPS[key][0]) == 1:
            export_movie(key)
else:
    print("Animations disabled. Set GENERATE_ANIMATIONS=True when ready.")


F_021: 51 finite saved frames
Timeline: 51 timestamps, 0 to 5; no interpolation
Exporting F_021_eps0.10_gamma-8_zeta0: 51 frames, 5.1 playback seconds
F_022: 51 finite saved frames
Timeline: 51 timestamps, 0 to 5; no interpolation
Exporting F_022_eps0.10_gamma-8_zeta-0.15: 51 frames, 5.1 playback seconds
F_023: 51 finite saved frames
Timeline: 51 timestamps, 0 to 5; no interpolation
Exporting F_023_eps0.10_gamma-8_zeta-0.3: 51 frames, 5.1 playback seconds
Passive: 51 finite saved frames
Timeline: 51 timestamps, 0 to 5; no interpolation
Exporting Passive_eps0.10_gamma0_zeta0: 51 frames, 5.1 playback seconds


### Synchronized comparison exports (two or three cases)


In [29]:
if GENERATE_ANIMATIONS:
    for key in ANIMATIONS_TO_EXPORT:
        if len(ANIMATION_GROUPS[key][0]) in (2, 3):
            export_movie(key)


### Output and playback verification

In [30]:
for path, frames, status in animation_outputs:
    print(f"{Path(path).name}: {frames} frames; {status}\n{path}")
if animation_outputs:
    from IPython.display import Video, Image, display
    for path, _, _ in animation_outputs:
        if path.endswith(".mp4"):
            display(Video(filename=path, embed=True, width=850))
        elif path.endswith(".gif"):
            display(Image(filename=path, width=850))
else:
    print("No animation exported in this session. MP4 readability is only verified after export.")


F_021_eps0.10_gamma-8_zeta0_directors.mp4: 51 frames; MP4 fully decoded; saved timeline frame count verified
/mnt/c/3.Programming Apps/python of the gauds/projects/Master thesis/FEniCS code/outputs/strain_coupling/figures/sweeps/animations/refined_F_tend5p0/local_directors_20261105_143828_835909Z/MP4/F_021_eps0.10_gamma-8_zeta0_directors.mp4
F_022_eps0.10_gamma-8_zeta-0.15_directors.mp4: 51 frames; MP4 fully decoded; saved timeline frame count verified
/mnt/c/3.Programming Apps/python of the gauds/projects/Master thesis/FEniCS code/outputs/strain_coupling/figures/sweeps/animations/refined_F_tend5p0/local_directors_20261105_143828_835909Z/MP4/F_022_eps0.10_gamma-8_zeta-0.15_directors.mp4
F_023_eps0.10_gamma-8_zeta-0.3_directors.mp4: 51 frames; MP4 fully decoded; saved timeline frame count verified
/mnt/c/3.Programming Apps/python of the gauds/projects/Master thesis/FEniCS code/outputs/strain_coupling/figures/sweeps/animations/refined_F_tend5p0/local_directors_20261105_143828_835909Z/MP4